In [2]:

import os, re, ssl, sys, subprocess, urllib.request as ur
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, Subset, DataLoader
import torchvision.transforms as T
import torchvision.models as tvm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

In [3]:
import numpy as np
from pathlib import Path

def read_any(path, dtype=float):
    path = Path(path)
    try:
        return np.load(path, allow_pickle=True)
    except Exception:
        return np.loadtxt(path, delimiter=';', dtype=dtype)

Read per-image annotations

In [4]:
# picking 0 as an example
# All four annotation files are expected to be named with this ID: 
# 0_exp.npy, 0_val.npy, 0_aro.npy, 0_lnd.npy under Dataset/annotations/
k = 0 
exp = int(read_any(f"Dataset/annotations/{k}_exp.npy", dtype=int))
val = float(read_any(f"Dataset/annotations/{k}_val.npy", dtype=float))
aro = float(read_any(f"Dataset/annotations/{k}_aro.npy", dtype=float))

# Landmarks can be 136 numbers(x1,y1,x2,y2,…) or 68x2
lnd_raw = read_any(f"Dataset/annotations/{k}_lnd.npy", dtype=float)

# incase lnd_raw is a list
lnd = np.asarray(lnd_raw)
if lnd.ndim == 1 and lnd.size == 136:
    lnd = lnd.reshape(68, 2)
elif lnd.ndim == 2 and lnd.shape in [(68, 2), (2, 68)]:
    if lnd.shape == (2, 68):
        lnd = lnd.T
else:
    raise ValueError(f"Unexpected landmarks shape: {lnd.shape}")

print("exp:", exp, "val:", val, "aro:", aro, "lnd shape:", lnd.shape)


exp: 0 val: -0.176846 aro: -0.0776398 lnd shape: (68, 2)


Read shared-array layout

In [5]:
import re
def load_annotations(root="Dataset"):
    root = Path(root)
    img_dir = root / "images"
    ann_dir = root / "annotations"
    assert img_dir.exists(), f"Missing images dir: {img_dir}"
    assert ann_dir.exists(), f"Missing annotations dir: {ann_dir}"

    if (ann_dir / "exp.npy").exists():
        exp = read_any(ann_dir / "exp.npy", dtype=int).astype(np.int64)
        val = read_any(ann_dir / "val.npy", dtype=float).astype(np.float32)
        aro = read_any(ann_dir / "aro.npy", dtype=float).astype(np.float32)

        lnd_raw = read_any(ann_dir / "lnd.npy", dtype=float)
        lnd = np.asarray(lnd_raw)
        if lnd.ndim == 2 and lnd.shape[1] == 136:
            lnd = lnd.reshape(-1, 68, 2)
        elif not (lnd.ndim == 3 and lnd.shape[1:] == (68, 2)):
            raise ValueError(f"Unexpected lnd shape: {lnd.shape}")
        ids = np.arange(len(exp), dtype=np.int64)
        return {"ids": ids, "exp": exp, "val": val, "aro": aro, "lnd": lnd}

    # discover numeric IDs from images like 0.jpg / 1.png
    ids = []
    for p in list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")):
        m = re.match(r"^(\d+)\.(jpg|png)$", p.name)
        if m:
            ids.append(int(m.group(1)))
    ids = sorted(ids)
    assert ids, "No numeric image filenames found under images/"

    N = len(ids)
    exp = np.empty(N, dtype=np.int64)
    val = np.empty(N, dtype=np.float32)
    aro = np.empty(N, dtype=np.float32)
    lnd_list = []

    for i, k in enumerate(ids):
        exp[i] = int(read_any(ann_dir / f"{k}_exp.npy", dtype=int))
        val[i] = float(read_any(ann_dir / f"{k}_val.npy", dtype=float))
        aro[i] = float(read_any(ann_dir / f"{k}_aro.npy", dtype=float))

        p_lnd = ann_dir / f"{k}_lnd.npy"
        if p_lnd.exists():
            lnd_k = np.asarray(read_any(p_lnd, dtype=float))
            if lnd_k.ndim == 1 and lnd_k.size == 136:
                lnd_k = lnd_k.reshape(68, 2)
            elif lnd_k.ndim == 2 and lnd_k.shape == (2, 68):
                lnd_k = lnd_k.T
            elif lnd_k.ndim == 2 and lnd_k.shape == (68, 2):
                pass
            else:
                raise ValueError(f"id {k}: unexpected landmarks shape {lnd_k.shape}")
        else:
            # fill missing landmarks with NaNs so stacking works
            lnd_k = np.full((68, 2), np.nan, dtype=np.float32)

        lnd_list.append(lnd_k.astype(np.float32))

    lnd = np.stack(lnd_list, axis=0)  # (N, 68, 2)
    return {"ids": np.asarray(ids, dtype=np.int64), "exp": exp, "val": val, "aro": aro, "lnd": lnd}

# ---- run it ----
ann = load_annotations("Dataset")
print("loaded:")
print("  ids:", ann["ids"].shape)
print("  exp:", ann["exp"].shape, ann["exp"].dtype)
print("  val:", ann["val"].shape, "range:", float(np.nanmin(ann["val"])), float(np.nanmax(ann["val"])))
print("  aro:", ann["aro"].shape, "range:", float(np.nanmin(ann["aro"])), float(np.nanmax(ann["aro"])))
print("  lnd:", ann["lnd"].shape)
print("first 5 ids:", ann["ids"][:5], "exp:", ann["exp"][:5])


loaded:
  ids: (3999,)
  exp: (3999,) int64
  val: (3999,) range: -0.9872239828109741 0.982384979724884
  aro: (3999,) range: -0.6666669845581055 0.9841269850730896
  lnd: (3999, 68, 2)
first 5 ids: [0 1 2 3 4] exp: [0 0 4 0 2]


visualize a small grid to confirm augmentation

In [9]:
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as T

# Build a small preview batch safely (bx may not exist yet)
try:
    if 'train_loader' in globals():
        bx, by = next(iter(train_loader))
    elif 'AffectFaceDataset' in globals():
        # fallback: lightweight loader directly from dataset class if defined
        preview_ds = AffectFaceDataset("Dataset", transform=T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]))
        preview_loader = DataLoader(preview_ds, batch_size=16, shuffle=False, num_workers=0)
        bx, by = next(iter(preview_loader))
    else:
        # ultimate fallback: read a few images directly from folder
        from pathlib import Path
        from PIL import Image
        img_dir = Path("Dataset")/"images"
        paths = sorted(list(img_dir.glob("*.jpg"))[:16] + list(img_dir.glob("*.png"))[:16])[:16]
        if not paths:
            raise RuntimeError("No images found under Dataset/images")
        tfm = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        tensors = []
        for p in paths:
            tensors.append(tfm(Image.open(p).convert("RGB")))
        import torch
        bx = torch.stack(tensors, dim=0)
except Exception as e:
    print("[warn] Could not build preview batch:", e)
    bx = None

if bx is not None:
    grid = torchvision.utils.make_grid(bx[:16], nrow=8, normalize=True)
    plt.figure(figsize=(12,4))
    plt.imshow(grid.permute(1,2,0))
    plt.axis("off")
    plt.show()
else:
    print("[info] Preview skipped.")

[warn] Could not build preview batch: name 'AffectFaceDataset' is not defined
[info] Preview skipped.


Main

Ensures secure HTTPS requests when PyTorch downloads pretrained weights, On some macOS systems, SSL certs break, this forces torchvision to use trusted certificates from the certifi package.

In [ ]:
try:
    import certifi
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "certifi"])
    import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ctx = ssl.create_default_context(cafile=certifi.where())
ur.install_opener(ur.build_opener(ur.HTTPSHandler(context=ctx)))

Annotations: Read .npy files or semicolon-separated text files

In [ ]:
# loads images and annotations, outputs tensors: (B, 3, 224, 224)
class AffectFaceDataset(Dataset):
    def __init__(self, root_dir="Dataset", image_size=224, transform=None):
        self.root = Path(root_dir)
        self.img_dir = self.root / "images"
        self.ann_dir = self.root / "annotations"
        assert self.img_dir.exists(), f"missing {self.img_dir}"
        assert self.ann_dir.exists(), f"missing {self.ann_dir}"

        # stores all numeric IDs from images like 0.jpg / 1.png
        # only stores the ids with 123.jpg/png format
        ids = []
        for p in list(self.img_dir.glob("*.jpg")) + list(self.img_dir.glob("*.png")):
            
            # extract numeric ID(123) from filename
            m = re.match(r"^(\d+)\.(jpg|png)$", p.name)
            if m:
                ids.append(int(m.group(1)))
                
        # sort the ids to have a consistent order
        self.ids = sorted(ids)
        assert self.ids, f"no images found in {self.img_dir}"

        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        self.transform = transform or T.Compose([
            T.Resize((image_size, image_size)), # Resize image to 224×224
            T.ToTensor(), # Convert image to tensor
            T.Normalize(mean, std) # Normalize with ImageNet stats
        ])

    # how many samples are in your dataset
    def __len__(self):
        return len(self.ids)

    def _read_annotations(self, k): 
        exp = int(read_any(self.ann_dir / f"{k}_exp.npy", dtype=int))
        val = float(read_any(self.ann_dir / f"{k}_val.npy", dtype=float))
        aro = float(read_any(self.ann_dir / f"{k}_aro.npy", dtype=float))
        return exp, val, aro # (exp, val, aro)

    # pytorch gives you idx, you use it to find the actual ID k (like 123) from your self.ids list
    def __getitem__(self, idx):
        k = self.ids[idx]
        # try jpg first, then png
        img_path = self.img_dir / f"{k}.jpg"
        if not img_path.exists():
            img_path = img_path.with_suffix(".png")

        # Open the image and force it into 3-channel RGB
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        exp, val, aro = self._read_annotations(k)
        targets = {
            "exp": torch.tensor(exp, dtype=torch.long),
            "val": torch.tensor(val, dtype=torch.float32),
            "aro": torch.tensor(aro, dtype=torch.float32),
        }
        return image, targets

# Uses StratifiedShuffleSplit to preserve class distribution:
def setup_data_splits(dataset_path="Dataset", batch_size=32, random_state=42):
    base_dataset = AffectFaceDataset(dataset_path, image_size=224)
    N = len(base_dataset)
    exp_labels = np.array([int(base_dataset[i][1]["exp"]) for i in range(N)]) # grabs each sample’s expression label
    idx_all = np.arange(N)

    # Train = 70%, Validation = 15%, Test = 15%
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=random_state)
    train_idx, temp_idx = next(sss1.split(idx_all, exp_labels))

    # Splits that 30% into Val and Test, each 15%.
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=random_state) 
    val_rel, test_rel = next(sss2.split(temp_idx, exp_labels[temp_idx]))
    val_idx, test_idx = temp_idx[val_rel], temp_idx[test_rel]

    print(f"Split sizes: train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

    # Train: resize + normalize plus augmentation (flip, rotate, jitter, affine) → prevents overfitting.
    # Purpose: Data augmentation.
    # It artificially creates variation in your training images so the model learns to generalize, not memorize.
    # Every time you load a training image, it might look slightly different (flipped, rotated, shifted, brighter/darker)
    train_transform = T.Compose([
        T.Resize((224, 224)),
        T.RandomHorizontalFlip(0.5),
        T.RandomRotation(10),
        T.ColorJitter(0.2, 0.2, 0.2, 0.1),
        T.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    # Val/Test: only resize + normalize → stable evaluation.
    eval_transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    # Each split uses the same images, but with different transforms (train vs eval).
    train_dataset = AffectFaceDataset(dataset_path, transform=train_transform)
    val_dataset   = AffectFaceDataset(dataset_path, transform=eval_transform)
    test_dataset  = AffectFaceDataset(dataset_path, transform=eval_transform)

    # train_idx, val_idx, test_idx were made earlier by StratifiedShuffleSplit
    train_set = Subset(train_dataset, train_idx)
    val_set   = Subset(val_dataset,   val_idx)
    test_set  = Subset(test_dataset,  test_idx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=0)

    class_dist = Counter(int(train_set.dataset[i][1]["exp"]) for i in train_set.indices)
    num_classes = 8
    counts = np.array([class_dist.get(c, 0) for c in range(num_classes)], dtype=np.float32)
    counts[counts == 0] = 1.0
    class_weights = (counts.sum() / (num_classes * counts)).astype(np.float32)

    return train_loader, val_loader, test_loader, class_weights

# one for classification (expressions),
# one for regression (valence/arousal).
class MultiTaskHead(nn.Module):
    def __init__(self, in_features, num_classes=8):
        super().__init__()

        # Takes the feature vector (say 512-D from ResNet18, or 1280-D from EfficientNet-B0).
        # Maps it directly into num_classes (here 8). This is your expression classifier.
        self.expression_head = nn.Linear(in_features, num_classes)
        self.regression_head = nn.Sequential(
            # Maps the same feature vector → 128-D hidden layer → 2 outputs
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 2),
            # Tanh squashes those 2 values into the range [-1, 1]
            nn.Tanh(),  
        )

    def forward(self, features):
        logits = self.expression_head(features)
        valaro = self.regression_head(features)
        return {"expression": logits, "valence": valaro[:, 0], "arousal": valaro[:, 1]}

# CrossEntropy for expressions. 
# MSE for valence + arousal (ignores -2 which means “missing label”)
class MultiTaskLoss(nn.Module):
    def __init__(self, class_weights=None, alpha=1.0, beta=0.5, gamma=0.5):
        super().__init__()
        # alpha = weight for expression classification.
        # beta = weight for valence regression.
        # gamma = weight for arousal regression.
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        if class_weights is not None:
            #  If your dataset is imbalanced (e.g. more “happy” than “fear”), you pass in class weights so rare classes count more.
            weight = torch.as_tensor(class_weights, dtype=torch.float32)
            self.register_buffer("ce_weight", weight)
        else:
            self.ce_weight = None
            
        # Cross-entropy for categorical expressions.
        # Mean-squared error for valence and arousal regression.
        self.ce_loss = nn.CrossEntropyLoss(weight=self.ce_weight)
        self.mse_loss = nn.MSELoss()

    def forward(self, preds, targets):
        exp_loss = self.ce_loss(preds["expression"], targets["exp"])
        # Some samples don’t have valence/arousal labels (marked -2) -> val
        val_mask = (targets["val"] != -2)
        if val_mask.any():
            val_loss = self.mse_loss(preds["valence"][val_mask], targets["val"][val_mask])
        else:
            val_loss = preds["valence"].new_tensor(0.0)

        aro_mask = (targets["aro"] != -2)
        if aro_mask.any():
            aro_loss = self.mse_loss(preds["arousal"][aro_mask], targets["aro"][aro_mask])
        else:
            aro_loss = preds["arousal"].new_tensor(0.0)

        total = self.alpha * exp_loss + self.beta * val_loss + self.gamma * aro_loss
        return {"total": total, "expression": exp_loss, "valence": val_loss, "arousal": aro_loss}

def _strip_classifier_and_get_dim(model, family: str):
    if family == "resnet":
        feat_dim = model.fc.in_features
        model.fc = nn.Identity()
    elif family == "efficientnet":
        feat_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
    else:
        raise ValueError(f"Unknown family {family}")
    return model, feat_dim

# loads the CNN backbone
# backbone = all convs + pooling → produces a flat feature vector.
def get_backbone(name: str, pretrained: bool):
    name = name.lower()
    if name == "resnet18":
        weights = tvm.ResNet18_Weights.DEFAULT if pretrained else None
        return _strip_classifier_and_get_dim(tvm.resnet18(weights=weights), "resnet")
    if name == "efficientnet_b0":
        weights = tvm.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        return _strip_classifier_and_get_dim(tvm.efficientnet_b0(weights=weights), "efficientnet")
    raise ValueError(f"Unsupported backbone name: {name}")

# “Give me ResNet18/EfficientNet-B0 backbone, chop off its classifier, attach my own MultiTaskHead.”
# GenericMultiTaskModel is the wrapper that glues backbone (feature extractor) + MultiTaskHead (expression + valence/arousal) together.
def ResNet18MultiHead(pretrained=True, num_classes=8):
    bb, fd = get_backbone("resnet18", pretrained)
    return GenericMultiTaskModel(bb, fd, num_classes)

def EfficientNetB0MultiHead(pretrained=True, num_classes=8):
    bb, fd = get_backbone("efficientnet_b0", pretrained)
    return GenericMultiTaskModel(bb, fd, num_classes)

class GenericMultiTaskModel(nn.Module):
    def __init__(self, backbone: nn.Module, feat_dim: int, num_classes: int = 8):
        super().__init__()
        self.backbone = backbone
        self.head = MultiTaskHead(feat_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x) # classification branch (expressions)
        return self.head(feats) # regression branch (valence, arousal)

# Epoch loops
def _train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    tot_loss, tot_samples, correct_exp = 0.0, 0, 0
    for images, targets in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}

        optimizer.zero_grad()
        preds = model(images)
        losses = criterion(preds, targets)
        losses["total"].backward()
        optimizer.step()

        bs = images.size(0)
        tot_samples += bs
        tot_loss += losses["total"].item() * bs
        correct_exp += (preds["expression"].argmax(1) == targets["exp"]).sum().item()
    return tot_loss / tot_samples, 100.0 * correct_exp / tot_samples

@torch.no_grad()
def _eval_epoch(model, loader, criterion, device):
    model.eval()
    tot_loss, tot_samples, correct_exp = 0.0, 0, 0
    val_se, aro_se, val_n, aro_n = 0.0, 0.0, 0, 0

    for images, targets in tqdm(loader, desc="Evaluating", leave=False):
        images = images.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}
        preds = model(images)
        losses = criterion(preds, targets)

        bs = images.size(0)
        tot_samples += bs
        tot_loss += losses["total"].item() * bs
        correct_exp += (preds["expression"].argmax(1) == targets["exp"]).sum().item()

        val_mask = (targets["val"] != -2)
        aro_mask = (targets["aro"] != -2)
        if val_mask.any():
            val_se += ((preds["valence"][val_mask] - targets["val"][val_mask]) ** 2).sum().item()
            val_n  += int(val_mask.sum())
        if aro_mask.any():
            aro_se += ((preds["arousal"][aro_mask] - targets["aro"][aro_mask]) ** 2).sum().item()
            aro_n  += int(aro_mask.sum())

    avg_loss = tot_loss / tot_samples
    exp_acc = 100.0 * correct_exp / tot_samples
    val_rmse = (val_se / val_n) ** 0.5 if val_n > 0 else float("nan")
    aro_rmse = (aro_se / aro_n) ** 0.5 if aro_n > 0 else float("nan")
    return avg_loss, exp_acc, val_rmse, aro_rmse

# Turns gradients off -> no backprop
@torch.no_grad()
def _compute_test_metrics(model, loader, device):
    model.eval()
    exp_true, exp_pred = [], []
    val_true, val_pred = [], []
    aro_true, aro_pred = [], []

    for images, targets in loader:
        images = images.to(device)
        preds = model(images)

        exp_pred.extend(preds["expression"].argmax(1).cpu().numpy())
        exp_true.extend(targets["exp"].numpy())

        vmask = (targets["val"].numpy() != -2)
        amask = (targets["aro"].numpy() != -2)
        if vmask.any():
            val_pred.extend(preds["valence"].cpu().numpy()[vmask])
            val_true.extend(targets["val"].numpy()[vmask])
        if amask.any():
            aro_pred.extend(preds["arousal"].cpu().numpy()[amask])
            aro_true.extend(targets["aro"].numpy()[amask])

    acc = accuracy_score(exp_true, exp_pred)
    f1m = f1_score(exp_true, exp_pred, average="macro")
    def _rmse(p, t):
        return float(np.sqrt(np.mean((np.array(p) - np.array(t)) ** 2))) if len(t) else float("nan")
    def _corr(p, t):
        if len(t) < 2:
            return float("nan")
        p, t = np.asarray(p), np.asarray(t)
        if np.std(p) == 0 or np.std(t) == 0:
            return float("nan")
        return float(np.corrcoef(p, t)[0,1])
    def _sagr(p, t):
        if not len(t):
            return float("nan")
        p, t = np.asarray(p), np.asarray(t)
        return float((np.sign(p) == np.sign(t)).mean())
    def _ccc(p, t):
        if len(t) < 2:
            return float("nan")
        p, t = np.asarray(p), np.asarray(t)
        mu_p, mu_t = p.mean(), t.mean()
        var_p, var_t = p.var(), t.var()
        cov = np.mean((p - mu_p)*(t - mu_t))
        return float((2*cov) / (var_p + var_t + (mu_p - mu_t)**2 + 1e-8))

    rmse_val = _rmse(val_pred, val_true)
    rmse_aro = _rmse(aro_pred, aro_true)
    corr_val = _corr(val_pred, val_true)
    corr_aro = _corr(aro_pred, aro_true)
    sagr_val = _sagr(val_pred, val_true)
    sagr_aro = _sagr(aro_pred, aro_true)
    ccc_val  = _ccc(val_pred, val_true)
    ccc_aro  = _ccc(aro_pred, aro_true)

    return {
        "acc": acc,
        "f1_macro": f1m,
        "rmse_val": rmse_val,
        "rmse_aro": rmse_aro,
        "corr_val": corr_val,
        "corr_aro": corr_aro,
        "sagr_val": sagr_val,
        "sagr_aro": sagr_aro,
        "ccc_val": ccc_val,
        "ccc_aro": ccc_aro,
    }

def run_model(name, model, train_loader, val_loader, test_loader,
              class_weights, epochs=10, device=None, lr=3e-4, wd=1e-4):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    criterion = MultiTaskLoss(class_weights=class_weights, alpha=1.0, beta=0.5, gamma=0.5).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_val_loss = float("inf")
    best_state = None

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_rmse": [], "aro_rmse": []}
    print(f"\n[{name}] Starting training...")
    for ep in range(epochs):
        tr_loss, tr_acc = _train_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc, rmse_v, rmse_a = _eval_epoch(model, val_loader, criterion, device)
        scheduler.step(va_loss)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["val_rmse"].append(rmse_v)
        history["aro_rmse"].append(rmse_a)

        print(f"[{name}] ep{ep:02d} | train {tr_loss:.4f}/{tr_acc:.2f}%  | val {va_loss:.4f}/{va_acc:.2f}%  rmse(v){rmse_v:.3f} rmse(a){rmse_a:.3f}")

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    metrics = _compute_test_metrics(model, test_loader, device)
    metrics["history"] = history
    return metrics

def bootstrap_data(dataset_path="Dataset", batch_size=32, random_state=42):
    return setup_data_splits(dataset_path, batch_size, random_state)


In [ ]:
# build loaders and class weights that run_with_fallback will use
train_loader, val_loader, test_loader, class_weights = bootstrap_data(
    dataset_path="Dataset", batch_size=32, random_state=42
)

# choose device once, reuse everywhere
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Split sizes: train=2799, val=600, test=600
Using device: cpu


In [ ]:
# Architecture sanity check (no training): builds models and runs a dummy forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224)

    m_res = ResNet18MultiHead(pretrained=False, num_classes=8)
    out_res = m_res(dummy)
    nparams_res = sum(p.numel() for p in m_res.parameters())
    print("ResNet18MultiHead built | params:", nparams_res,
          "| outputs:", {k: tuple(v.shape) if hasattr(v, 'shape') else type(v) for k, v in out_res.items()})

    m_eff = EfficientNetB0MultiHead(pretrained=False, num_classes=8)
    out_eff = m_eff(dummy)
    nparams_eff = sum(p.numel() for p in m_eff.parameters())
    print("EfficientNetB0MultiHead built | params:", nparams_eff,
          "| outputs:", {k: tuple(v.shape) if hasattr(v, 'shape') else type(v) for k, v in out_eff.items()})



In [ ]:
# Evaluation helpers, plots, qualitative examples (no auto-generated report)
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for images, targets in loader:
        images = images.to(device)
        logits = model(images)["expression"].cpu()
        y_pred.extend(logits.argmax(1).numpy().tolist())
        y_true.extend(targets["exp"].numpy().tolist())
    return np.array(y_true), np.array(y_pred)

# Confusion matrix plot
def plot_confusion(y_true, y_pred, class_names=None, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names if class_names is not None else np.arange(cm.shape[1]),
                yticklabels=class_names if class_names is not None else np.arange(cm.shape[0]))
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); plt.show()

# Qualitative grid: shows k correct and k incorrect examples
@torch.no_grad()
def show_qualitative(model, loader, class_names=None, k=8, device=None):
    model.eval()
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    images_batch, targets_batch = next(iter(loader))
    images = images_batch.to(device)
    preds = model(images)
    pred_labels = preds["expression"].argmax(1).cpu().numpy()
    true_labels = targets_batch["exp"].numpy()

    correct_idx = np.where(pred_labels == true_labels)[0][:k]
    wrong_idx   = np.where(pred_labels != true_labels)[0][:k]
    sel = np.concatenate([correct_idx, wrong_idx])

    denorm = T.Normalize(mean=[-m/s for m, s in zip([0.485,0.456,0.406],[0.229,0.224,0.225])],
                         std=[1/s for s in [0.229,0.224,0.225]])

    n = len(sel)
    cols = min(8, n)
    rows = int(np.ceil(n/cols))
    plt.figure(figsize=(2.2*cols, 2.2*rows))
    for i, idx in enumerate(sel, 1):
        plt.subplot(rows, cols, i)
        img = denorm(images_batch[idx]).clamp(0,1)
        plt.imshow(img.permute(1,2,0))
        pl, tl = int(pred_labels[idx]), int(true_labels[idx])
        title = f"P:{pl} T:{tl}" if class_names is None else f"P:{class_names[pl]} T:{class_names[tl]}"
        color = "green" if pl==tl else "red"
        plt.title(title, color=color, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Plot training curves from metrics["history"]
def plot_history(history, title_prefix=""):
    if not history:
        print("No history to plot.")
        return
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.plot(epochs, history["train_loss"], label="train")
    plt.plot(epochs, history["val_loss"], label="val")
    plt.title(f"{title_prefix}Loss")
    plt.xlabel("epoch"); plt.legend()

    plt.subplot(1,3,2)
    plt.plot(epochs, history["train_acc"], label="train")
    plt.plot(epochs, history["val_acc"], label="val")
    plt.title(f"{title_prefix}Accuracy")
    plt.xlabel("epoch"); plt.legend()

    plt.subplot(1,3,3)
    plt.plot(epochs, history["val_rmse"], label="valence RMSE")
    plt.plot(epochs, history["aro_rmse"], label="arousal RMSE")
    plt.title(f"{title_prefix}RMSE (valence/arousal)")
    plt.xlabel("epoch"); plt.legend()
    plt.tight_layout()
    plt.show()

# Build comparison table (no auto report)

def summarize_results(results_dict):
    rows = []
    for name, m in results_dict.items():
        rows.append({
            "model": name,
            "acc": m.get("acc", float("nan")),
            "f1_macro": m.get("f1_macro", float("nan")),
            "rmse_val": m.get("rmse_val", float("nan")),
            "rmse_aro": m.get("rmse_aro", float("nan")),
            "corr_val": m.get("corr_val", float("nan")),
            "corr_aro": m.get("corr_aro", float("nan")),
            "sagr_val": m.get("sagr_val", float("nan")),
            "sagr_aro": m.get("sagr_aro", float("nan")),
            "ccc_val": m.get("ccc_val", float("nan")),
            "ccc_aro": m.get("ccc_aro", float("nan")),
        })
    df = pd.DataFrame(rows).set_index("model").round(3)
    display(df)
    return df



In [ ]:
def run_with_fallback(name, model_ctor, **kwargs):
    try:
        # try pretrained=True
        model = model_ctor(pretrained=True, **kwargs)
        return run_model(name, model, train_loader, val_loader, test_loader,
                         class_weights, epochs=10, device=device)
    except Exception as e:
        print(f"[{name}] pretrained download failed → fallback to pretrained=False\n{e}\n")
        model = model_ctor(pretrained=False, **kwargs)
        return run_model(name, model, train_loader, val_loader, test_loader,
                         class_weights, epochs=10, device=device)

m1 = run_with_fallback("resnet18",        ResNet18MultiHead,    num_classes=8)
m2 = run_with_fallback("efficientnet_b0", EfficientNetB0MultiHead, num_classes=8)

print("\nSUMMARY")
for name, m in [("resnet18", m1), ("efficientnet_b0", m2)]:
    print(f"{name:16s}  acc {m['acc']:.3f}  f1 {m['f1_macro']:.3f}  "
          f"rmse_val {m['rmse_val']:.3f}  rmse_aro {m['rmse_aro']:.3f}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/4star/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:11<00:00, 4.20MB/s]


[resnet18] ep00 | train 2.0167/28.94%  | val 1.8506/36.33%  rmse(v)0.416 rmse(a)0.363  lr 3.00e-04


[resnet18] ep01 | train 1.6779/41.26%  | val 1.6775/41.83%  rmse(v)0.426 rmse(a)0.338  lr 3.00e-04


[resnet18] ep02 | train 1.4974/50.02%  | val 1.8626/41.83%  rmse(v)0.397 rmse(a)0.337  lr 3.00e-04


...

[efficientnet_b0] ep05 | train 0.9334/69.17%  | val 1.8829/42.67%  rmse(v)0.405 rmse(a)0.352  lr 3.00e-04


[efficientnet_b0] ep06 | train 0.7642/75.10%  | val 1.9409/43.50%  rmse(v)0.403 rmse(a)0.355  lr 1.50e-04


[efficientnet_b0] ep07 | train 0.5750/82.92%  | val 1.9290/43.50%  rmse(v)0.390 rmse(a)0.354  lr 1.50e-04


[efficientnet_b0] ep08 | train 0.4381/87.39%  | val 2.0644/44.17%  rmse(v)0.393 rmse(a)0.348  lr 1.50e-04


[efficientnet_b0] ep09 | train 0.3786/89.28%  | val 2.0286/46.33%  rmse(v)0.390 rmse(a)0.347  lr 1.50e-04
[efficientnet_b0] TEST -> acc 0.503  f1 0.500  RMSE(val) 0.387  RMSE(aro) 0.341

SUMMARY
resnet18          acc 0.480  f1 0.471  rmse_val 0.400  rmse_aro 0.330
efficientnet_b0   acc 0.503  f1 0.500  rmse_val 0.387  rmse_aro 0.341


In [ ]:
# 1) Summary table
results = {"resnet18": m1, "efficientnet_b0": m2}
df = summarize_results(results)

# 2) Confusion matrix (optional): requires a trained model object; skip if not in memory
try:
    if 'resnet18_trained_model' in globals():
        y_t, y_p = collect_predictions(resnet18_trained_model.to(device), test_loader, device)
        plot_confusion(y_t, y_p, class_names=list(range(8)), title="ResNet18 Confusion Matrix")
    elif 'efficientnet_b0_trained_model' in globals():
        y_t, y_p = collect_predictions(efficientnet_b0_trained_model.to(device), test_loader, device)
        plot_confusion(y_t, y_p, class_names=list(range(8)), title="EfficientNet-B0 Confusion Matrix")
    else:
        print("[info] No model object in memory for confusion matrix; skipping.")
except Exception as e:
    print("[warn] Confusion matrix failed:", e)

# 3) Training curves (if history captured after edits)
for name, m in results.items():
    hist = m.get("history")
    if hist:
        plot_history(hist, title_prefix=f"{name} ")
    else:
        print(f"[info] No training history stored for {name}. Curves skipped. Run training cell again to capture.")

# 4) Qualitative examples (if trained model objects are available)
try:
    candidate_models = []
    if 'resnet18_trained_model' in globals():
        candidate_models.append(("resnet18", resnet18_trained_model))
    if 'efficientnet_b0_trained_model' in globals():
        candidate_models.append(("efficientnet_b0", efficientnet_b0_trained_model))
    if candidate_models:
        for nm, mdl in candidate_models:
            print(f"Qualitative examples for {nm}")
            show_qualitative(mdl.to(device), test_loader, class_names=list(range(8)), k=8, device=device)
    else:
        print("[info] No trained model objects found in memory. Qualitative grid skipped.")
except Exception as e:
    print("[warn] Qualitative visualization failed:", e)


In [ ]:
# m1 and m2 came from your run_with_fallback calls
print("resnet18     acc:", m1["acc"], " f1_macro:", m1["f1_macro"],
      " rmse_val:", m1["rmse_val"], " rmse_aro:", m1["rmse_aro"])

print("efficient_b0 acc:", m2["acc"], " f1_macro:", m2["f1_macro"],
      " rmse_val:", m2["rmse_val"], " rmse_aro:", m2["rmse_aro"])


resnet18     acc: 0.48  f1_macro: 0.4710918092958022  rmse_val: 0.3996772155170319  rmse_aro: 0.3296029502753119
efficient_b0 acc: 0.5033333333333333  f1_macro: 0.4998888630574332  rmse_val: 0.3869799534541992  rmse_aro: 0.3407388504539036
